<a href="https://colab.research.google.com/github/dhaviesayo/salford_nlp/blob/add-coreapi/APIs/core_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### CORE-API
core api provides access to the largest collection of thousands open access research papers from different providers.

CORE collects, harmonises and enriches large quantities of both metadata and full text research articles from thousands of data providers. On top of this continuously growing corpus, its provides a truly unique API providing real-time machine access to both the metadata and full texts of research papers, enabling developers to build and run innovative applications on top of CORE's content.


### How to run:
-  Run the `get_data()` function

### Params
- topics: list of topics
- num_results: number of results to output for the topic
- output_dir: optional output directory to write the output to as json
- api_key : optional api key, Get Apikey from `https://core.ac.uk/services/api`


### Usage

run example code :


data = get_data(topics=["sport", "entertainment"] , num_results= 10)


print(data)













In [ ]:
!pip install tenacity --upgrade
!pip install pandas

In [ ]:
import requests
from typing import List , Dict
import re
import pandas as pd
from tenacity import retry , stop_after_attempt , wait_exponential_jitter , RetryError

In [ ]:
@retry(stop=stop_after_attempt(10),wait=wait_exponential_jitter(5 , 15))  #retry if we hit ratelimit
def _send_request(endpoint: str ,topic:str , num_results:int ,api_key: str|None = None):
  res  =  requests.post(endpoint ,
                        headers={"Authorization": f"Bearer {api_key}"} if api_key else None,
                        json={"q": topic ,  "limit": num_results})
  if res.status_code == 200:
    return res.json()
  else:
    raise RuntimeError(f"couldnt access data for {topic}, finished with status code : {res.status_code}")

def send_request(endpoint: str ,topic:str , num_results:int ,api_key: str|None = None):
  try:
    result = _send_request(endpoint,topic,num_results,api_key)
    return result
  except RetryError as e:
      raise(e)

def process_data(entry: Dict) -> List:   #process and pick the field with better contextual information
  def _pick_best_field(child_entry:Dict)-> str:
      if not child_entry.get("abstract"):
        return  child_entry.get("fullText")[:2000]  #pick first 2000 words if 'abstract' field is empty
      url_pattern = r"^(https?:\/\/)?([\w\-]+\.)+[\w\-]+(\/[\w\-._~:/?#[\]@!$&'()*+,;=]*)?$"
      is_url = bool(re.match(url_pattern,child_entry.get("abstract")))
      if is_url:
        return  child_entry.get("fullText")[:2000]   #pick first 2000 words if 'abstract' field contains url
      else:
        return child_entry.get("abstract")   #pick 'abstract' only if its present and its not a url
  out =  list(map(_pick_best_field , entry.get("results"))) #run with map for perform transformation in parallel
  return out


def get_data(topics: List , num_results: int ,
             output_dir:str|None = None ,api_key : str|None = None)-> Dict[str,List]:
  """
  topics : list of topics to search for
  num_results: number of results to return
  output_dir: useful if building a dataset, to store result to dataset
  api_key: api key from -> https://core.ac.uk/services/api
  """
  endpoint = "https://api.core.ac.uk/v3/search/works"
  results =  {}
  for topic in topics:
    results[topic] = process_data(send_request(endpoint,topic,num_results,api_key))
  if output_dir:
     if output_dir.endswith(".csv"):
        pd.DataFrame.from_dict(results).to_csv(output_dir)
     else:
        raise RuntimeError("`output_dir` isnt a .csv path")

  return results


In [ ]:
#usage

data = get_data(["sport"] , 10)
data